# SAM 2 vs DeepLabV3-ResNet50 — Comparaison

Ce notebook compare :
- **DeepLabV3-ResNet50** (fine-tuné sur Oxford Pet, val IoU=0.82)
- **SAM 2.1 Hiera Large** (zero-shot, prompted + automatique)

Sur :
- **Oxford-IIIT Pet** test set (IoU, Dice, Pixel Accuracy)
- **Images Reddit** (qualitatif, hors distribution)

> Kernel : **`bcs_analysis`** (Python 3.10 + PyTorch 2.5 + SAM2 1.1)

## 1. Configuration & imports

In [ ]:
%matplotlib inline

import sys
import os
from pathlib import Path

ROOT = Path("/home/SPeillet/bcs_determination")
sys.path.insert(0, str(ROOT / "src"))
os.environ["TORCH_HOME"] = str(ROOT / "checkpoints")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
from torchvision import transforms
from torchvision.transforms import functional as TF
from torch.utils.data import DataLoader
from scipy import ndimage as ndi

from bcs_pipeline.lightning_module.segmentation_module import (
    LitSegmentationModule,
    transforms_inv_normalize,
)
from bcs_pipeline.data.oxford_segmentation_datamodule import OxfordSegmentationDataset

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
CHECKPOINT_PATH = (
    ROOT / "experiments/deeplabv3_resnet50_adam_cosine_annealing/checkpoints"
    / "epoch=epoch=30-val_iou=val/iou=0.82-step=6417.ckpt"
)
DATA_DIR = ROOT / "data/Oxford-IIIT_pet_dataset"
IMAGE_SIZE = 256
NUM_CLASSES = 3
CLASS_NAMES = ["Pet (foreground)", "Background", "Border"]
PALETTE = [(0, 0, 0), (255, 255, 255), (128, 128, 128)]
MAX_PROMPTED = 100
MAX_AUTO = 30
BORDER_WIDTH = 5

assert CHECKPOINT_PATH.exists(), f"Checkpoint introuvable : {CHECKPOINT_PATH}"
assert DATA_DIR.exists(), f"Dataset introuvable : {DATA_DIR}"
print(f"Checkpoint : {CHECKPOINT_PATH.name}")
print(f"Dataset    : {DATA_DIR}")

## 2. Fonctions utilitaires

In [ ]:
def mask_to_rgb(mask, palette=None):
    if palette is None:
        palette = PALETTE
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_idx, colour in enumerate(palette):
        if cls_idx < len(palette):
            rgb[mask == cls_idx] = colour
    return rgb


def compute_metrics(pred_mask, gt_mask, num_classes=3):
    class_iou, class_dice, class_pixel_acc = [], [], []
    for c in range(num_classes):
        pred_c = pred_mask == c
        gt_c = gt_mask == c
        inter = (pred_c & gt_c).sum()
        union = (pred_c | gt_c).sum()
        class_iou.append(inter / (union + 1e-8))
        class_dice.append(2 * inter / (pred_c.sum() + gt_c.sum() + 1e-8))
        class_pixel_acc.append(inter / (gt_c.sum() + 1e-8))
    pixel_acc = (pred_mask == gt_mask).sum() / gt_mask.size
    return {
        "class_iou": class_iou, "mean_iou": np.mean(class_iou),
        "class_dice": class_dice, "mean_dice": np.mean(class_dice),
        "class_pixel_acc": class_pixel_acc, "pixel_accuracy": pixel_acc,
    }


def sam2_mask_to_trimap(sam2_mask, border_width=5):
    sam2_mask = np.asarray(sam2_mask, dtype=bool)
    trimap = np.ones(sam2_mask.shape, dtype=np.int64)
    trimap[sam2_mask] = 0
    eroded = ndi.binary_erosion(sam2_mask, iterations=border_width)
    border = sam2_mask & ~eroded
    trimap[border] = 2
    return trimap


def resize_mask(mask_np, size):
    return np.array(
        Image.fromarray(mask_np.astype(np.int8)).resize((size, size), Image.NEAREST)
    )

## 3. Chargement des modeles

In [ ]:
print("Chargement DeepLabV3-ResNet50 …")
deeplab = LitSegmentationModule.load_from_checkpoint(
    str(CHECKPOINT_PATH), map_location="cpu",
)
deeplab.eval().to(device).freeze()
print(f"  OK — {sum(p.numel() for p in deeplab.parameters()):,} params")

dl_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
print("Chargement SAM 2.1 Hiera Large …")
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

ckpt = ROOT / "checkpoints" / "sam2.1_hiera_large.pt"
if not ckpt.exists():
    print("  Telechargement (~900 MB) …")
    import urllib.request
    ckpt.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt",
        str(ckpt),
    )
sam2_model = build_sam2("configs/sam2.1/sam2.1_hiera_l.yaml", str(ckpt), device=device)
predictor = SAM2ImagePredictor(sam2_model)
auto_gen = SAM2AutomaticMaskGenerator(
    model=sam2_model, points_per_side=32, points_per_batch=64,
    pred_iou_thresh=0.7, stability_score_thresh=0.92,
)
print(f"  OK — {sum(p.numel() for p in sam2_model.parameters()):,} params")

In [ ]:
def sam2_prompted(pil_img):
    img_np = np.array(pil_img.convert("RGB"))
    w, h = pil_img.size
    predictor.set_image(img_np)
    masks, scores, _ = predictor.predict(
        point_coords=np.array([[w // 2, h // 2]]),
        point_labels=np.array([1]),
        multimask_output=True,
    )
    best = masks[np.argmax(scores)]
    return np.array(Image.fromarray(best).resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)).astype(bool)


def sam2_automatic(pil_img):
    img_np = np.array(pil_img.convert("RGB"))
    w, h = pil_img.size
    centre = np.array([w / 2, h / 2])
    all_masks = auto_gen.generate(img_np)
    if not all_masks:
        return np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=bool)
    best_m, best_d = None, float("inf")
    for m in all_masks:
        ys, xs = np.where(m["segmentation"])
        if len(xs) == 0:
            continue
        d = np.hypot(xs.mean() - centre[0], ys.mean() - centre[1])
        if d < best_d:
            best_d, best_m = d, m["segmentation"]
    if best_m is None:
        best_m = all_masks[0]["segmentation"]
    return np.array(Image.fromarray(best_m).resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)).astype(bool)

## 4. Preparation du jeu de test Oxford

In [ ]:
ds_raw = OxfordSegmentationDataset(data_dir=str(DATA_DIR), mode="test")
n_total = len(ds_raw)
rng = np.random.RandomState(42)
prompted_idx = rng.choice(n_total, min(MAX_PROMPTED, n_total), replace=False)
auto_idx = prompted_idx[:min(MAX_AUTO, MAX_PROMPTED)]
print(f"Test: {n_total} | Prompted: {len(prompted_idx)} | Auto: {len(auto_idx)}")

## 5. Evaluation DeepLabV3 (jeu de test complet)

In [ ]:
dl_mask_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=TF.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])
ds_full = OxfordSegmentationDataset(
    data_dir=str(DATA_DIR), mode="test",
    transform=dl_transforms, mask_transform=dl_mask_tf,
)
loader_full = DataLoader(ds_full, batch_size=16, num_workers=4, shuffle=False)

confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.int64)
dl_all_ious = []

with torch.no_grad():
    for imgs, masks in tqdm(loader_full, desc="DeepLabV3"):
        imgs, masks = imgs.to(device), masks.to(device).long()
        preds = torch.argmax(deeplab(imgs), dim=1)
        for cp in range(NUM_CLASSES):
            for cg in range(NUM_CLASSES):
                confusion[cp, cg] += ((preds == cp) & (masks == cg)).sum().item()
        for b in range(preds.shape[0]):
            p, m = preds[b].cpu().numpy(), masks[b].cpu().numpy()
            ious = []
            for c in range(NUM_CLASSES):
                inter = ((p == c) & (m == c)).sum()
                union = ((p == c) | (m == c)).sum()
                ious.append(inter / (union + 1e-8))
            dl_all_ious.append(np.mean(ious))

dl_class_iou, dl_class_dice, dl_class_pxa = [], [], []
for c in range(NUM_CLASSES):
    tp = confusion[c, c].item()
    fp = confusion[c, :].sum().item() - tp
    fn = confusion[:, c].sum().item() - tp
    tot = confusion[:, c].sum().item()
    dl_class_iou.append(tp / (tp + fp + fn + 1e-8))
    dl_class_dice.append(2 * tp / (2 * tp + fp + fn + 1e-8))
    dl_class_pxa.append(tp / (tot + 1e-8))
dl_pa = confusion.diag().sum().item() / confusion.sum().item()

deeplab_metrics = {
    "pixel_accuracy": dl_pa,
    "mean_iou": np.mean(dl_class_iou), "mean_dice": np.mean(dl_class_dice),
    "class_iou": dl_class_iou, "class_dice": dl_class_dice,
    "class_pixel_acc": dl_class_pxa, "all_ious": np.array(dl_all_ious),
}
print(f"mIoU={np.mean(dl_class_iou):.4f}  mDice={np.mean(dl_class_dice):.4f}  PixAcc={dl_pa:.4f}")

## 6. Evaluation SAM 2 (prompted + automatique)

In [ ]:
print(f"SAM 2 prompted ({len(prompted_idx)} images) …")
s2p_ious, s2p_dices, s2p_pa = [], [], []
s2p_ci = [[] for _ in range(NUM_CLASSES)]

for idx in tqdm(prompted_idx, desc="SAM2 prompted"):
    img_path, mask_path = ds_raw.samples[idx]
    pil = Image.open(img_path).convert("RGB")
    gt = resize_mask(np.array(Image.open(mask_path), dtype=np.int64) - 1, IMAGE_SIZE)
    try:
        mask = sam2_prompted(pil)
        trimap = sam2_mask_to_trimap(mask, BORDER_WIDTH)
    except Exception:
        trimap = np.ones((IMAGE_SIZE, IMAGE_SIZE), dtype=np.int64)
    m = compute_metrics(trimap, gt, NUM_CLASSES)
    s2p_ious.append(m["mean_iou"]); s2p_dices.append(m["mean_dice"]); s2p_pa.append(m["pixel_accuracy"])
    for c in range(NUM_CLASSES):
        s2p_ci[c].append(m["class_iou"][c])

s2p_metrics = {
    "pixel_accuracy": np.mean(s2p_pa),
    "mean_iou": np.mean(s2p_ious), "mean_dice": np.mean(s2p_dices),
    "class_iou": [np.mean(c) for c in s2p_ci],
    "class_dice": [2 * np.mean(s2p_ci[c]) / (1 + np.mean(s2p_ci[c])) for c in range(NUM_CLASSES)],
}
print(f"mIoU={s2p_metrics['mean_iou']:.4f}  mDice={s2p_metrics['mean_dice']:.4f}  PixAcc={s2p_metrics['pixel_accuracy']:.4f}")

In [ ]:
print(f"SAM 2 automatique ({len(auto_idx)} images) …")
s2a_ious, s2a_dices, s2a_pa = [], [], []
s2a_ci = [[] for _ in range(NUM_CLASSES)]

for idx in tqdm(auto_idx, desc="SAM2 auto"):
    img_path, mask_path = ds_raw.samples[idx]
    pil = Image.open(img_path).convert("RGB")
    gt = resize_mask(np.array(Image.open(mask_path), dtype=np.int64) - 1, IMAGE_SIZE)
    try:
        mask = sam2_automatic(pil)
        trimap = sam2_mask_to_trimap(mask, BORDER_WIDTH)
    except Exception:
        trimap = np.ones((IMAGE_SIZE, IMAGE_SIZE), dtype=np.int64)
    m = compute_metrics(trimap, gt, NUM_CLASSES)
    s2a_ious.append(m["mean_iou"]); s2a_dices.append(m["mean_dice"]); s2a_pa.append(m["pixel_accuracy"])
    for c in range(NUM_CLASSES):
        s2a_ci[c].append(m["class_iou"][c])

s2a_metrics = {
    "pixel_accuracy": np.mean(s2a_pa),
    "mean_iou": np.mean(s2a_ious), "mean_dice": np.mean(s2a_dices),
    "class_iou": [np.mean(c) for c in s2a_ci],
    "class_dice": [2 * np.mean(s2a_ci[c]) / (1 + np.mean(s2a_ci[c])) for c in range(NUM_CLASSES)],
}
print(f"mIoU={s2a_metrics['mean_iou']:.4f}  mDice={s2a_metrics['mean_dice']:.4f}  PixAcc={s2a_metrics['pixel_accuracy']:.4f}")

## 7. Tableau comparatif

In [ ]:
rows = []
metrics_all = [deeplab_metrics, s2p_metrics, s2a_metrics]
for name, met in [("DeepLabV3 (fine-tune)", deeplab_metrics),
                  ("SAM 2.1 Prompted (zero-shot)", s2p_metrics),
                  ("SAM 2.1 Automatic (zero-shot)", s2a_metrics)]:
    row = {"Modele": name, "mIoU": met["mean_iou"], "mDice": met["mean_dice"], "Pixel Acc": met["pixel_accuracy"]}
    for c, cn in enumerate(CLASS_NAMES):
        row[f"IoU {cn}"] = met["class_iou"][c]
        row[f"Dice {cn}"] = met["class_dice"][c]
    rows.append(row)

df = pd.DataFrame(rows)
print("\n" + "=" * 90)
print("  COMPARAISON — Segmentation Oxford-IIIT Pet")
print("=" * 90)
print(df.to_string(index=False, float_format="%.4f"))
print("=" * 90)

## 8. Graphiques comparatifs

In [ ]:
models = ["DeepLabV3\n(fine-tune)", "SAM 2.1\n(prompted)", "SAM 2.1\n(automatic)"]
colors = ["#2196F3", "#4CAF50", "#FF9800"]
metric_keys = {"mIoU": "mean_iou", "mDice": "mean_dice", "Pixel Acc": "pixel_accuracy"}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for j, (ax, mname) in enumerate(zip(axes, ["mIoU", "mDice", "Pixel Acc"])):
    vals = [m[metric_keys[mname]] for m in metrics_all]
    bars = ax.bar(models, vals, color=colors, edgecolor="black", linewidth=0.5)
    ax.set_ylim(0, 1.0)
    ax.set_title(mname, fontsize=14, fontweight="bold")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{v:.4f}", ha="center", va="bottom", fontsize=12, fontweight="bold")
fig.suptitle("Metriques globales", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(6 * NUM_CLASSES, 5))
for c, (ax, cn) in enumerate(zip(axes, CLASS_NAMES)):
    iou_v = [m["class_iou"][c] for m in metrics_all]
    dice_v = [m["class_dice"][c] for m in metrics_all]
    x = np.arange(3)
    w = 0.35
    ax.bar(x - w / 2, iou_v, w, label="IoU", color="#2196F3", alpha=0.8)
    ax.bar(x + w / 2, dice_v, w, label="Dice", color="#4CAF50", alpha=0.8)
    ax.set_ylim(0, 1.05); ax.set_title(cn, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(models, fontsize=8); ax.legend()
fig.suptitle("IoU & Dice par classe", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
cats = ["mIoU", "mDice", "Pixel Acc"] + [f"IoU\n{n}" for n in CLASS_NAMES]
N = len(cats)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
for name, met, col in [("DeepLabV3", deeplab_metrics, "#2196F3"),
                       ("SAM 2.1 Prompted", s2p_metrics, "#4CAF50"),
                       ("SAM 2.1 Auto", s2a_metrics, "#FF9800")]:
    vals = [met["mean_iou"], met["mean_dice"], met["pixel_accuracy"]] + list(met["class_iou"])
    vals += vals[:1]
    ax.plot(angles, vals, "o-", lw=2, label=name, color=col)
    ax.fill(angles, vals, alpha=0.1, color=col)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(cats, fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_title("Radar — Oxford-IIIT Pet", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(s2p_ious, bins=30, alpha=0.6, label="SAM 2 Prompted", color="#4CAF50")
axes[0].hist(dl_all_ious[:len(s2p_ious)], bins=30, alpha=0.6, label="DeepLabV3", color="#2196F3")
axes[0].set_title("Distribution mIoU par image"); axes[0].legend()
axes[1].hist(s2p_dices, bins=30, alpha=0.6, label="SAM 2 Prompted", color="#4CAF50")
axes[1].set_title("Distribution mDice par image"); axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Visualisations qualitatives — Oxford

In [ ]:
vis_idx = prompted_idx[:6]
fig, axes = plt.subplots(len(vis_idx), 5, figsize=(20, 4 * len(vis_idx)))
cols = ["Image", "Ground Truth", "DeepLabV3", "SAM 2 Prompted", "SAM 2 Auto"]
for c, t in enumerate(cols):
    axes[0, c].set_title(t, fontsize=12, fontweight="bold")

for r, idx in enumerate(vis_idx):
    img_path, mask_path = ds_raw.samples[idx]
    pil = Image.open(img_path).convert("RGB")
    gt = resize_mask(np.array(Image.open(mask_path), dtype=np.int64) - 1, IMAGE_SIZE)
    disp = np.array(pil.resize((IMAGE_SIZE, IMAGE_SIZE)))

    axes[r, 0].imshow(disp); axes[r, 0].axis("off")
    axes[r, 1].imshow(mask_to_rgb(gt)); axes[r, 1].axis("off")

    with torch.no_grad():
        t = dl_transforms(pil).unsqueeze(0).to(device)
        dl_pred = torch.argmax(deeplab(t), dim=1).squeeze().cpu().numpy()
    axes[r, 2].imshow(mask_to_rgb(dl_pred)); axes[r, 2].axis("off")

    s2p = sam2_prompted(pil)
    s2p_t = sam2_mask_to_trimap(s2p, BORDER_WIDTH)
    axes[r, 3].imshow(mask_to_rgb(s2p_t)); axes[r, 3].axis("off")

    s2a = sam2_automatic(pil)
    s2a_t = sam2_mask_to_trimap(s2a, BORDER_WIDTH)
    axes[r, 4].imshow(mask_to_rgb(s2a_t)); axes[r, 4].axis("off")

    dl_m = compute_metrics(dl_pred, gt)
    sp_m = compute_metrics(s2p_t, gt)
    sa_m = compute_metrics(s2a_t, gt)
    axes[r, 0].set_ylabel(
        f"DL={dl_m['mean_iou']:.2f} S2P={sp_m['mean_iou']:.2f} S2A={sa_m['mean_iou']:.2f}",
        fontsize=9, rotation=0, labelpad=100,
    )
plt.tight_layout()
plt.show()

## 10. Test sur les images Reddit

In [ ]:
reddit_dir = ROOT / "data/Reddit_example"
reddit_imgs = sorted(reddit_dir.glob("*.webp"))
print(f"Images Reddit: {len(reddit_imgs)}")
for p in reddit_imgs:
    print(f"  -> {p.name}")

In [ ]:
inv_norm = transforms_inv_normalize()
fig, axes = plt.subplots(len(reddit_imgs), 6, figsize=(24, 5 * len(reddit_imgs)))
if len(reddit_imgs) == 1:
    axes = axes[np.newaxis, :]
cols = ["Image", "DeepLabV3", "Overlay DL", "SAM 2 Prompted", "SAM 2 Auto", "Overlay SAM2"]
for c, t in enumerate(cols):
    axes[0, c].set_title(t, fontsize=11, fontweight="bold")

for r, ip in enumerate(reddit_imgs):
    pil = Image.open(ip).convert("RGB")
    disp = np.array(pil.resize((IMAGE_SIZE, IMAGE_SIZE)))
    axes[r, 0].imshow(disp); axes[r, 0].axis("off")
    axes[r, 0].set_ylabel(ip.name[:35], fontsize=8, rotation=0, labelpad=100)

    with torch.no_grad():
        t = dl_transforms(pil).unsqueeze(0).to(device)
        dl_pred = torch.argmax(deeplab(t), dim=1).squeeze().cpu().numpy()
    axes[r, 1].imshow(mask_to_rgb(dl_pred)); axes[r, 1].axis("off")

    vis = inv_norm(dl_transforms(pil))
    ov = vis.permute(1, 2, 0).numpy()
    rgb_pred = mask_to_rgb(dl_pred).astype(np.float32) / 255
    axes[r, 2].imshow(np.clip(0.6 * ov + 0.4 * rgb_pred, 0, 1)); axes[r, 2].axis("off")

    s2p = sam2_prompted(pil)
    s2p_t = sam2_mask_to_trimap(s2p, BORDER_WIDTH)
    axes[r, 3].imshow(mask_to_rgb(s2p_t)); axes[r, 3].axis("off")

    s2a = sam2_automatic(pil)
    s2a_t = sam2_mask_to_trimap(s2a, BORDER_WIDTH)
    axes[r, 4].imshow(mask_to_rgb(s2a_t)); axes[r, 4].axis("off")

    ov2 = disp.copy().astype(np.float32) / 255
    green = np.zeros_like(ov2)
    green[s2p.astype(bool)] = [0, 1, 0]
    axes[r, 5].imshow(np.clip(0.6 * ov2 + 0.4 * green, 0, 1)); axes[r, 5].axis("off")

    dl_pct = [(dl_pred == c).sum() / dl_pred.size * 100 for c in range(NUM_CLASSES)]
    s2p_pct = [(s2p_t == c).sum() / s2p_t.size * 100 for c in range(NUM_CLASSES)]
    print(f"\n{ip.name}:")
    print(f"  DeepLabV3: Pet={dl_pct[0]:.1f}% BG={dl_pct[1]:.1f}% Border={dl_pct[2]:.1f}%")
    print(f"  SAM2 Prom: Pet={s2p_pct[0]:.1f}% BG={s2p_pct[1]:.1f}% Border={s2p_pct[2]:.1f}%")

plt.tight_layout()
plt.show()

## 11. Resume final

In [ ]:
print("\n" + "=" * 60)
print("  RESUME FINAL")
print("=" * 60)
print(f"  DeepLabV3 (fine-tune)        : mIoU={deeplab_metrics['mean_iou']:.4f}  mDice={deeplab_metrics['mean_dice']:.4f}  PixAcc={deeplab_metrics['pixel_accuracy']:.4f}")
print(f"  SAM 2.1 Prompted (zero-shot) : mIoU={s2p_metrics['mean_iou']:.4f}  mDice={s2p_metrics['mean_dice']:.4f}  PixAcc={s2p_metrics['pixel_accuracy']:.4f}")
print(f"  SAM 2.1 Auto (zero-shot)     : mIoU={s2a_metrics['mean_iou']:.4f}  mDice={s2a_metrics['mean_dice']:.4f}  PixAcc={s2a_metrics['pixel_accuracy']:.4f}")
print("=" * 60)
print()
print("Points cles :")
print("  - DeepLabV3 domine car il est fine-tune sur Oxford Pet (3 classes natives)")
print("  - SAM 2 est zero-shot : pas d'entrainement sur Oxford Pet")
print("  - Le border est le point faible de SAM 2 (approximation par erosion)")
print("  - SAM 2 reste tres bon pour isoler le foreground sans aucun fine-tuning")

## 12. Explorateur interactif par race

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import re

# --- Build breed → sample indices mapping from ds_raw (test set) ---
breed_to_indices = {}
for i, (img_path, _) in enumerate(ds_raw.samples):
    fname = os.path.basename(img_path).replace(".jpg", "")
    breed = re.sub(r"_\d+$", "", fname)
    breed_to_indices.setdefault(breed, []).append(i)

breed_names = sorted(breed_to_indices.keys())

# --- Widgets ---
breed_dropdown = widgets.Dropdown(
    options=breed_names,
    description="Race :",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="350px"),
)

image_dropdown = widgets.Dropdown(
    description="Image :",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="350px"),
)

run_button = widgets.Button(
    description="Lancer l'inférence",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="200px"),
)

output_area = widgets.Output(layout=widgets.Layout(width="100%"))


def _update_image_list(*_):
    """Update image dropdown when breed changes."""
    breed = breed_dropdown.value
    indices = breed_to_indices[breed]
    labels = []
    for idx in indices:
        fname = os.path.basename(ds_raw.samples[idx][0])
        labels.append((fname, idx))
    image_dropdown.options = labels
    if labels:
        image_dropdown.value = labels[0][1]


def _run_inference(_):
    """Run DeepLabV3 + SAM2 on the selected image and display results."""
    idx = image_dropdown.value
    if idx is None:
        return

    with output_area:
        clear_output(wait=True)
        img_path, mask_path = ds_raw.samples[idx]
        pil_img = Image.open(img_path).convert("RGB")

        # Ground truth trimap
        gt = resize_mask(
            np.array(Image.open(mask_path), dtype=np.int64) - 1, IMAGE_SIZE
        )
        disp = np.array(pil_img.resize((IMAGE_SIZE, IMAGE_SIZE)))

        # DeepLabV3 inference
        with torch.no_grad():
            t = dl_transforms(pil_img).unsqueeze(0).to(device)
            dl_pred = torch.argmax(deeplab(t), dim=1).squeeze().cpu().numpy()

        # SAM 2 prompted inference
        try:
            s2p_mask = sam2_prompted(pil_img)
            s2p_trimap = sam2_mask_to_trimap(s2p_mask, BORDER_WIDTH)
        except Exception as e:
            print(f"SAM2 error: {e}")
            s2p_trimap = np.ones((IMAGE_SIZE, IMAGE_SIZE), dtype=np.int64)

        # Metrics
        dl_m = compute_metrics(dl_pred, gt)
        s2p_m = compute_metrics(s2p_trimap, gt)

        # --- Plot ---
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        titles = [
            "Image originale",
            "Ground Truth",
            f"DeepLabV3 (mIoU={dl_m['mean_iou']:.3f})",
            f"SAM 2 Prompted (mIoU={s2p_m['mean_iou']:.3f})",
        ]
        images_to_show = [disp, mask_to_rgb(gt), mask_to_rgb(dl_pred), mask_to_rgb(s2p_trimap)]

        for ax, img, title in zip(axes, images_to_show, titles):
            ax.imshow(img)
            ax.set_title(title, fontsize=11, fontweight="bold")
            ax.axis("off")

        breed = breed_dropdown.value
        fname = os.path.basename(img_path)
        fig.suptitle(f"{breed} — {fname}", fontsize=14, fontweight="bold", y=1.02)
        plt.tight_layout()
        plt.show()


breed_dropdown.observe(_update_image_list, names="value")
_update_image_list()  # initialize image list

run_button.on_click(_run_inference)

display(widgets.VBox([
    widgets.HBox([breed_dropdown, image_dropdown, run_button]),
    output_area,
]))